# Import Library

In [1]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report

# Library Khusus Bahasa Indonesia
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Import Data

In [2]:
file_path = "../../data/synthetic_report_dataset.csv" # Pastikan nama/path file-nya benar

# 1. Kita intip dulu file mentahnya untuk melihat pemisahnya
with open(file_path, 'r', encoding='utf-8') as f:
    baris_pertama = f.readline()
    print("Isi baris pertama file mentah:\n👉", baris_pertama)

# 2. Auto-Deteksi Pemisah Kolom
if '|' in baris_pertama:
    pemisah = '|'
else:
    pemisah = ','
print(f"Sistem mendeteksi pemisah kolom: '{pemisah}'\n")

# 3. Load Data
nama_kolom = ['teks_keluhan_awam', 'teks_laporan_teknisi', 'kategori_aset', 'severity', 'root_cause', 'tindakan']
df = pd.read_csv(file_path, sep=pemisah, names=nama_kolom, on_bad_lines='skip')

print(f"Baris sebelum dibersihkan: {df.shape[0]}")

# 4. Pembersihan Data Lembut (Soft Cleaning)
# Hanya buang baris kalau KEDUA kolom teksnya kosong (bukan salah satu)
df = df.dropna(subset=['teks_keluhan_awam', 'teks_laporan_teknisi'], how='all')

# Jika baris pertama ternyata adalah header (mengandung kata 'teks_keluhan'), buang baris itu
df = df[df['teks_keluhan_awam'].astype(str).str.contains('teks_keluhan', case=False) == False]

# Reset nomor urut baris
df = df.reset_index(drop=True)

# Pastikan jadi string agar tidak error saat digabung
df['teks_keluhan_awam'] = df['teks_keluhan_awam'].astype(str)
df['teks_laporan_teknisi'] = df['teks_laporan_teknisi'].astype(str)

# 5. Gabungkan Teks (Sebagai Input X)
df['input_teks'] = df['teks_keluhan_awam'] + " " + df['teks_laporan_teknisi']

# 6. MENGAMBIL KOLOM TARGET (Sebagai Kunci Jawaban Y)
kolom_target = ['kategori_aset', 'severity', 'root_cause', 'tindakan']
Y = df[kolom_target]

print(f"Total data SIAP DITRAINING: {df.shape[0]}")
print(f"Bentuk Input (X): 1 Kolom Teks Gabungan")
print(f"Bentuk Target (Y): {Y.shape[1]} Kolom Kunci Jawaban")
print("\n--- Intip Isi Target (Y) ---")
display(Y.head(3)) # Menampilkan 3 baris pertama dari kolom target

Isi baris pertama file mentah:
👉 AC di ruang meeting lantai 2 netes air terus nih,"Saluran pembuangan air kondensasi tersumbat lumut dan debu, sudah dibersihkan tuntas",HVAC,Sedang,Tersumbat,Pembersihan

Sistem mendeteksi pemisah kolom: ','

Baris sebelum dibersihkan: 666
Total data SIAP DITRAINING: 666
Bentuk Input (X): 1 Kolom Teks Gabungan
Bentuk Target (Y): 4 Kolom Kunci Jawaban

--- Intip Isi Target (Y) ---


,kategori_aset,severity,root_cause,tindakan
0,HVAC,Sedang,Tersumbat,Pembersihan
1,Pompa Air,Tinggi,Keausan,Penggantian Part
2,Mesin Produksi,Tinggi,Tersumbat,Pembersihan


# Cleaning & Preprocessing

In [3]:
stemmer = StemmerFactory().create_stemmer()
stopword_remover = StopWordRemoverFactory().create_stop_word_remover()

def clean_preprocessing(text):
    # a. Case Folding (Lowercase)
    text = text.lower()
    # b. Cleaning (Hapus karakter spesial & angka)
    text = re.sub(r'[^a-z\s]', '', text)
    # c. Stopword Removal
    text = stopword_remover.remove(text)
    # d. Stemming (Mengubah ke kata dasar)
    text = stemmer.stem(text)
    return text

print("Sedang melakukan Preprocessing teks (ini agak lama)...")
df['clean_teks'] = df['input_teks'].apply(clean_preprocessing)

Sedang melakukan Preprocessing teks (ini agak lama)...


# Train-Test Split

In [4]:
# Pisahkan sebelum Augmentasi untuk mencegah Data Leakage!
X = df['clean_teks']
y = df[kolom_target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# DATA AUGMENTATION

In [5]:
def synonym_augmentation(text):
    # Contoh sederhana Synonym Replacement
    synonyms = {"panas": "overheat", "rusak": "kendala", "bocor": "netes", "mati": "padam"}
    words = text.split()
    augmented_words = [synonyms.get(w, w) for w in words]
    return " ".join(augmented_words)

# Menambah jumlah data latih 2x lipat dengan augmentasi
X_train_aug = X_train.apply(synonym_augmentation)
X_train_final = pd.concat([X_train, X_train_aug])
y_train_final = pd.concat([y_train, y_train])

print(f"Data setelah Augmentasi: {X_train_final.shape[0]} baris.")

Data setelah Augmentasi: 1064 baris.


# FEATURE ENGINEERING & MODELING

In [7]:
# Menggunakan Pipeline agar proses TF-IDF dan RF menjadi satu kesatuan
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('clf', MultiOutputClassifier(RandomForestClassifier(random_state=42)))
])

# Definisi Grid Parameter untuk Tuning
param_grid = {
    'tfidf__max_features': [1000, 2000],
    'clf__estimator__n_estimators': [100, 200],
    'clf__estimator__max_depth': [None, 10, 20]
}

print("Mulai Hyperparameter Tuning dengan GridSearchCV...")
grid_search = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=1, verbose=1)
grid_search.fit(X_train_final, y_train_final)

print(f"\nBest Parameters: {grid_search.best_params_}")

Mulai Hyperparameter Tuning dengan GridSearchCV...
Fitting 3 folds for each of 12 candidates, totalling 36 fits

Best Parameters: {'clf__estimator__max_depth': None, 'clf__estimator__n_estimators': 100, 'tfidf__max_features': 1000}


# EVALUATION

In [8]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

print("\n=== HASIL EVALUASI FINAL ===")

exact_match_manual = (y_test.values == y_pred).all(axis=1).mean()
print(f"Exact Match Ratio (Benar Semua 4 Kolom): {exact_match_manual * 100:.2f}%")

# Menghitung Akurasi Per Kolom (Tetap menggunakan accuracy_score biasa)
print("\n--- Akurasi Individu per Target ---")
for i, col in enumerate(kolom_target):
    # Mengambil 1 kolom dari y_test (kunci jawaban) dan 1 kolom dari y_pred (tebakan)
    acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
    print(f"Akurasi {col:15}: {acc * 100:.2f}%")


=== HASIL EVALUASI FINAL ===
Exact Match Ratio (Benar Semua 4 Kolom): 85.07%

--- Akurasi Individu per Target ---
Akurasi kategori_aset  : 100.00%
Akurasi severity       : 88.81%
Akurasi root_cause     : 97.76%
Akurasi tindakan       : 95.52%
